# Análise Causal — Impacto da Precipitação nos Atrasos de Voos

## Objetivo

Estimar o efeito causal da precipitação no aeroporto de origem sobre o atraso de partida dos voos no Brasil.

A análise busca responder:

> Mantidas comparáveis as demais características do voo, quanto o aumento da precipitação altera o atraso esperado na partida?

## Definição do problema causal

### Tratamento (T)

Precipitação (`precipitation`), medida em mm/h no aeroporto de origem durante a hora programada da partida.

### Outcome (Y)

Atraso de partida (`departure_delay_minutes`), medido em minutos.

### População

Voos realizados entre 2022 e 2025, considerando atrasos entre -120 e 720 minutos.

### Unidade de análise

Um voo programado.

## Potenciais confundidores

Variáveis consideradas conhecidas antes da realização do voo e que podem estar relacionadas tanto às condições de exposição quanto ao atraso:

- aeroporto de origem;
- aeroporto de destino;
- companhia aérea;
- hora programada da partida;
- dia da semana;
- mês;
- ano.

Outras características operacionais serão avaliadas antes da especificação final do modelo.

## Variáveis que não serão utilizadas como controles

Variáveis observadas após ou durante a realização do voo, ou diretamente derivadas do outcome, não serão utilizadas como controles:

- horário real de partida;
- horário real de chegada;
- atraso de chegada;
- situação da partida;
- situação da chegada;
- justificativa do atraso.

A inclusão dessas variáveis poderia introduzir leakage ou viés pós-tratamento.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_parquet(
    "../data/processed/flight_weather.parquet"
)

In [ ]:
df_causal = df[
    (df["flight_status"] == "REALIZADO")
    & (df["departure_delay_minutes"].between(-120, 720))
].copy()

In [ ]:
df_causal["hour"] = df_causal[
    "scheduled_departure_local"
].dt.hour

df_causal["month"] = df_causal[
    "scheduled_departure_local"
].dt.month

df_causal["day_of_week"] = df_causal[
    "scheduled_departure_local"
].dt.dayofweek

df_causal["year"] = df_causal[
    "scheduled_departure_local"
].dt.year

In [ ]:
print(f"Observações: {len(df_causal):,}")

df_causal[
    [
        "departure_delay_minutes",
        "precipitation",
        "origin_airport",
        "destination_airport",
        "airline",
        "hour",
        "day_of_week",
        "month",
        "year",
    ]
].isna().sum()

In [ ]:
categorical_features = [
    "origin_airport",
    "destination_airport",
    "airline",
]

df_causal[categorical_features].nunique()

In [ ]:
for col in categorical_features:
    print(f"\n{col}")
    print(df_causal[col].value_counts().head(10))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


categorical_features = [
    "origin_airport",
    "hour",
    "month",
]

X = df_causal[categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_features,
        )
    ]
)

X_encoded = preprocessor.fit_transform(X)

In [ ]:
X_encoded.shape

In [ ]:
print(
    f"Non-zero values: {X_encoded.nnz:,}")

In [ ]:
Y = df_causal["departure_delay_minutes"].to_numpy()

D = df_causal["precipitation"].to_numpy()

X = X_encoded

In [ ]:
print("X:", X.shape)
print("Y:", Y.shape)
print("D:", D.shape)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
)

model_y = Ridge(alpha=1.0)

model_y.fit(X_train, y_train)

y_pred = model_y.predict(X_test)

print(
    f"R² outcome model: {r2_score(y_test, y_pred):.4f}"
)

In [ ]:
X_train, X_test, d_train, d_test = train_test_split(
    X,
    D,
    test_size=0.2,
    random_state=42,
)

model_d = Ridge(alpha=1.0)

model_d.fit(X_train, d_train)

d_pred = model_d.predict(X_test)

print(
    f"R² treatment model: {r2_score(d_test, d_pred):.4f}"
)

In [ ]:
categorical_features_b = [
    "origin_airport",
    "destination_airport",
    "airline",
    "hour",
    "day_of_week",
    "month",
    "year",
]

In [ ]:
X_b = df_causal[categorical_features_b]

preprocessor_b = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_features_b,
        )
    ]
)

X_b_encoded = preprocessor_b.fit_transform(X_b)

In [ ]:
X_b_encoded.shape

In [ ]:
print(f"Non-zero values: {X_b_encoded.nnz:,}")

In [ ]:
X_b_encoded.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_b_encoded,
    Y,
    test_size=0.2,
    random_state=42,
)

model_y_b = Ridge(alpha=1.0)
model_y_b.fit(X_train, y_train)

y_pred_b = model_y_b.predict(X_test)

print(
    f"R² outcome model B: {r2_score(y_test, y_pred_b):.4f}"
)

In [ ]:
X_train, X_test, d_train, d_test = train_test_split(
    X_b_encoded,
    D,
    test_size=0.2,
    random_state=42,
)

model_d_b = Ridge(alpha=1.0)
model_d_b.fit(X_train, d_train)

d_pred_b = model_d_b.predict(X_test)

print(
    f"R² treatment model B: {r2_score(d_test, d_pred_b):.4f}"
)

In [ ]:
import doubleml as dml
import pandas as pd
import scipy.sparse as sp

In [ ]:
print(X_b_encoded.shape)
print(X_b_encoded.dtype)

memory_sparse = (
    X_b_encoded.data.nbytes
    + X_b_encoded.indices.nbytes
    + X_b_encoded.indptr.nbytes
)

print(
    f"Memória sparse: {memory_sparse / 1024**2:.2f} MB"
)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

sample_idx = rng.choice(
    X_b_encoded.shape[0],
    size=500_000,
    replace=False,
)

X_sample = X_b_encoded[sample_idx]
Y_sample = Y[sample_idx]
D_sample = D[sample_idx]

In [ ]:
rng = np.random.default_rng(42)

sample_idx = rng.choice(
    X_b_encoded.shape[0],
    size=100_000,]]
    replace=False,
)

X_sample = X_b_encoded[sample_idx].toarray()
Y_sample = Y[sample_idx]
D_sample = D[sample_idx]

print(X_sample.shape)
print(f"{X_sample.nbytes / 1024**2:.2f} MB")

In [ ]:
import doubleml as dml
from sklearn.linear_model import Ridge

dml_data = dml.DoubleMLData.from_arrays(
    X_sample,
    Y_sample,
    D_sample,
)

ml_l = Ridge(alpha=1.0)
ml_m = Ridge(alpha=1.0)

dml_plr = dml.DoubleMLPLR(
    dml_data,
    ml_l=ml_l,
    ml_m=ml_m,
    n_folds=3,
)

dml_plr.fit()

dml_plr.summary

In [ ]:
results = []

for seed in [42, 123, 456, 789, 2026]:

    rng = np.random.default_rng(seed)

    sample_idx = rng.choice(
        X_b_encoded.shape[0],
        size=100_000,
        replace=False,
    )

    X_sample = X_b_encoded[sample_idx].toarray()
    Y_sample = Y[sample_idx]
    D_sample = D[sample_idx]

    dml_data = dml.DoubleMLData.from_arrays(
        X_sample,
        Y_sample,
        D_sample,
    )

    model = dml.DoubleMLPLR(
        dml_data,
        ml_l=Ridge(alpha=1.0),
        ml_m=Ridge(alpha=1.0),
        n_folds=3,
    )

    model.fit()

    results.append({
        "seed": seed,
        "coef": model.coef[0],
        "se": model.se[0],
    })

pd.DataFrame(results)